### This notebook features a baseline XGBoost model.

In [1]:
import kagglehub
import pandas as pd
from pathlib import Path

from sklearn.metrics import classification_report
from xgboost import XGBClassifier

/opt/anaconda3/envs/cryptofraud/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_raw_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.Dataframe]:
    """
    1. Fetch the eliptic dataset from kagglehub.
    2. Load the feature, edges, and classes CSVs into DataFrames.
    """
    path = kagglehub.dataset_download("ellipticco/elliptic-data-set")
    # print("Path to dataset files:", path)
    data_dir = Path(path) / "elliptic_bitcoin_dataset"

    features_path = data_dir / "elliptic_txs_features.csv"
    edges_path = data_dir / "elliptic_txs_edgelist.csv"
    classes_path = data_dir / "elliptic_txs_classes.csv"

    features = pd.read_csv(features_path, header=None)
    edges = pd.read_csv(edges_path)
    classes = pd.read_csv(classes_path)

    return features, edges, classes

In [3]:
def build_node_table(features: pd.DataFrame, classes: pd.Dataframe) -> pd.DataFrame:
    """
    Merge the features and classes table to create a node table with labels
    Column 0: Transaction id (txId)
    Column 1: time_step (1-49)
    Column 2-166: Remaining numeric features
    """
    
    # rename features
    features = features.copy()
    features.rename(columns={0: 'txId', 1: 'time_step'}, inplace=True)

    # Join with labels on transaction id
    nodes = features.merge(classes, on='txId', how="left")
    return nodes

In [4]:
def preprocess_data(nodes: pd.DataFrame) -> tuple(pd.DataFrame, pd.Series, pd.Series):
    """
    Prepare for supervised learning:
    Drop unlabeled transactions, split features and label, etc.
    Returns X (features), y (labels), and time_step (for time-based splitting)
    """
    # Keep only labelled transactions
    labelled = nodes[nodes["class"] != "unknown"].copy()

    # Map string labels to ints for computation
    label_map = {"1" : 1, "2" : 0} # 1 = illicit, 0 = licit
    y = labelled["class"].map(label_map)

    # Save feature columns in X. Feature columns are all columns with integer labels (non-string)
    features = [c for c in labelled.columns if isinstance(c, int)]
    X = labelled[features].copy()

    # Keep time_step for time-based splitting
    time_step = labelled["time_step"].copy()

    return X, y, time_step


In [18]:
def train_xgboost(X: pd.DataFrame, y: pd.Series, time_step: pd.Series):
    """
    Train simple XGBoost baseline with time-based split and print evaluation.
    
    Time-based splits:
    - Train: time_step <= 34
    - Val: 35 <= time_step <= 41
    - Test: time_step >= 42
    
    Returns:
        model: Trained XGBoost classifier
    """

    # Time-based splits
    train_mask = time_step <= 34
    val_mask = (time_step >= 35) & (time_step <= 41)
    test_mask = time_step >= 42

    X_train = X[train_mask]
    y_train = y[train_mask]
    
    X_val = X[val_mask]
    y_val = y[val_mask]
    
    X_test = X[test_mask]
    y_test = y[test_mask]

    print(f"Train set size: {len(X_train)} (time_step <= 34)")
    print(f"Val set size: {len(X_val)} (time_step 35-41)")
    print(f"Test set size: {len(X_test)} (time_step >= 42)")

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=200, 
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        tree_method="hist",
    )

    model.fit(X_train, y_train)

    # Evaluate on validation set
    # y_val_pred = model.predict(X_val)
    # print("\nXGBoost Baseline Evaluation - Validation Set")
    # print(classification_report(y_val, y_val_pred, target_names=["licit (0)", "illicit (1)"]))

    # Evaluate on test set
    y_test_pred = model.predict(X_test)
    print("\nXGBoost Baseline Evaluation - Test Set")
    print(classification_report(y_test, y_test_pred, target_names=["licit (0)", "illicit (1)"]))
    
    return model

In [19]:
def main():
    features, edges, classes = load_raw_data()

    nodes = build_node_table(features, classes)

    print("\nClass distribution (including 'unknown'):")
    print(nodes["class"].value_counts())

    X, y, time_step = preprocess_data(nodes)

    print("\nLabelled samples:", len(y))
    print("Class distribution (labelled only):")
    print(y.value_counts(normalize=True))


    model = train_xgboost(X, y, time_step)
    return model

model = main()


Class distribution (including 'unknown'):
class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64

Labelled samples: 46564
Class distribution (labelled only):
class
0    0.902392
1    0.097608
Name: proportion, dtype: float64
Train set size: 29894 (time_step <= 34)
Val set size: 7829 (time_step 35-41)
Test set size: 8841 (time_step >= 42)

XGBoost Baseline Evaluation - Test Set
              precision    recall  f1-score   support

   licit (0)       0.97      0.99      0.98      8433
 illicit (1)       0.80      0.47      0.59       408

    accuracy                           0.97      8841
   macro avg       0.89      0.73      0.79      8841
weighted avg       0.97      0.97      0.97      8841



In [7]:
# Save the trained model to the project directory
import os

# Create models directory if it doesn't exist
models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

# Save the model using the booster's save_model method

model_path = models_dir / "xgboost_baseline.json"
model.get_booster().save_model(str(model_path))
# print(f"Model saved to: {model_path.absolute()}")

### A supervised XGBoost baseline achieves high performance on the Elliptic dataset, which indicates that the engineered tabular features already capture sufficient information about a transaction. However, this baseline treats transactions as independent, ignores unlabeled nodes, and relies on static features, making it poorly suited for a real-time inference system. We therefore continue with a GNN not to maximize metrics, but to model transaction risk as a property of a graph, leveraging the unlabeled structure and enabling real-time (realistic) streaming detection.